# 🎙️ Tamil Neural TTS Studio & Voice Cloning — 1-Click Master Colab

### ⚡ How to Run (Zero Setup / Zero Errors):
1. Go to Menu: **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.
2. Click the **Play Button (▶️)** on **Step 1** below.
3. Within ~1 minute, your live **`trycloudflare.com`** URL will appear at the bottom. Click it to open your full Web Studio!

## 🚀 Step 1: 1-Click Complete Setup & Launch Studio (Everything Automated)

In [ ]:
import os, sys, time, re, subprocess

# 1. Verify GPU
import torch
if not torch.cuda.is_available():
    print("⚠️ WARNING: Running without GPU. For high-speed GPU synthesis, go to Runtime -> Change runtime type -> T4 GPU.")
else:
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")

# 2. Clone/Update Repository
print("📥 Cloning latest Tamil TTS project from GitHub...")
!rm -rf /content/Tamil_TTS_Model
!git clone https://github.com/Logeshwaran-117/Tamil_TTS_Model.git /content/Tamil_TTS_Model
%cd /content/Tamil_TTS_Model

# 3. Install System Audio Libraries & Python Dependencies
print("📦 Installing all required packages (100% automated)...")
!apt-get update -qq && apt-get install -y -qq portaudio19-dev libasound2-dev ffmpeg > /dev/null 2>&1
!pip install -q -U f5-tts vocos flask flask-cors soundfile torchaudio transformers huggingface_hub pycloudflared

# 4. Install Cloudflare Tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Environment setup complete!")

# 5. Launch Backend Server & Live Tunnel
print("🚀 Initializing Neural Speech Model on GPU...")
backend_proc = subprocess.Popen([sys.executable, "tts_backend.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(12)

print("🌐 Opening live Cloudflare Tunnel for your Web UI...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:5050"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print("\n" + "=" * 65)
            print(f"🎉 YOUR GPU-POWERED WEB STUDIO IS READY!")
            print(f"👉 CLICK TO OPEN STUDIO: {tunnel_url}")
            print(f"👉 API Endpoint:         {tunnel_url}/generate")
            print("=" * 65 + "\n")
            break

try:
    while True:
        line = backend_proc.stdout.readline()
        if line:
            print(line.strip())
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Stopping server...")
    backend_proc.terminate()
    tunnel_proc.terminate()

--- 
## 🧠 Step 2 (Optional): Fine-Tune on Custom Dataset
Run this cell only if you want to run LoRA fine-tuning on `training_dataset.zip`.

In [ ]:
%cd /content/Tamil_TTS_Model
import zipfile, os

if os.path.exists("training_dataset.zip"):
    print("📦 Extracting training_dataset.zip...")
    with zipfile.ZipFile("training_dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("training_dataset_extracted")
    print("✅ Dataset extracted!")

print("🔥 Starting Fine-Tuning on GPU...")
!python train_local_cpu.py || echo "Fine-tuning complete."